# Análise exploratóriaObjetivo: entender a qualidade do dado **antes** de responder qualquer pergunta,e transformar cada anomalia encontrada em uma regra explícita no pipeline.Toda consulta aqui roda sobre a camada **trusted**, que ainda não tem filtronenhum. É de propósito: a exploração precisa enxergar a sujeira.

## 1. Volumetria e cobertura da janela

In [ ]:
%sqlSELECT 'yellow' AS trip_type, ref_year, ref_month, COUNT(*) AS corridasFROM ifood_case.trusted.ny_taxi_trip_yellow GROUP BY ALLUNION ALLSELECT 'green', ref_year, ref_month, COUNT(*)FROM ifood_case.trusted.ny_taxi_trip_green GROUP BY ALLORDER BY trip_type, ref_year, ref_month;

## 2. Perfil das colunas de interesseNulos, mínimos, máximos e cardinalidade das cinco colunas exigidas pelo case.

In [ ]:
%sqlSELECT    COUNT(*)                                                     AS total_corridas,    COUNT(*) - COUNT(VendorID)                                   AS vendorid_nulos,    COUNT(DISTINCT VendorID)                                     AS vendorid_distintos,    COUNT(*) - COUNT(passenger_count)                            AS passenger_count_nulos,    MIN(passenger_count)                                         AS passenger_count_min,    MAX(passenger_count)                                         AS passenger_count_max,    COUNT(*) - COUNT(total_amount)                               AS total_amount_nulos,    ROUND(MIN(total_amount), 2)                                  AS total_amount_min,    ROUND(MAX(total_amount), 2)                                  AS total_amount_max,    ROUND(AVG(total_amount), 2)                                  AS total_amount_medio,    MIN(tpep_pickup_datetime)                                    AS pickup_mais_antigo,    MAX(tpep_pickup_datetime)                                    AS pickup_mais_recenteFROM ifood_case.trusted.ny_taxi_trip_yellow;

### Achado 1 — datas impossíveisO `pickup_mais_antigo` e o `pickup_mais_recente` acima quase certamente caem**fora** de Jan–Mai/2023. Os arquivos da TLC carregam registros com taxímetrodesconfigurado (datas de 2001, 2008, 2090...).Isso tem consequência prática direta: um `GROUP BY mês do pickup` sem filtroproduz linhas fantasma no resultado. A consulta abaixo mostra o tamanho do problema.

In [ ]:
%sqlSELECT    date_format(tpep_pickup_datetime, 'yyyy-MM') AS mes_do_pickup,    COUNT(*) AS corridasFROM ifood_case.trusted.ny_taxi_trip_yellowGROUP BY ALLORDER BY corridas DESC;

### Achado 2 — mês do arquivo ≠ mês do eventoCruzando a partição de competência com o mês real do embarque fica visívelquanto de cada arquivo está "fora do lugar". É o que justifica ter separado`ref_year`/`ref_month` (arquivo) de `pickup_year`/`pickup_month` (evento).

In [ ]:
%sqlSELECT    concat(ref_year, '-', ref_month)             AS mes_do_arquivo,    date_format(tpep_pickup_datetime, 'yyyy-MM') AS mes_do_pickup,    COUNT(*)                                     AS corridasFROM ifood_case.trusted.ny_taxi_trip_yellowGROUP BY ALLHAVING concat(ref_year, '-', ref_month) <> date_format(tpep_pickup_datetime, 'yyyy-MM')ORDER BY corridas DESCLIMIT 30;

### Achado 3 — duração da corrida

In [ ]:
%sqlWITH duracoes AS (    SELECT (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0 AS minutos    FROM ifood_case.trusted.ny_taxi_trip_yellow    UNION ALL    SELECT (unix_timestamp(lpep_dropoff_datetime) - unix_timestamp(lpep_pickup_datetime)) / 60.0    FROM ifood_case.trusted.ny_taxi_trip_green)SELECT    CASE        WHEN minutos IS NULL   THEN '0 · nula'        WHEN minutos < 0       THEN '1 · negativa'        WHEN minutos = 0       THEN '2 · zerada'        WHEN minutos < 1       THEN '3 · menos de 1 min'        WHEN minutos < 5       THEN '4 · 1 a 5 min'        WHEN minutos < 15      THEN '5 · 5 a 15 min'        WHEN minutos < 30      THEN '6 · 15 a 30 min'        WHEN minutos < 60      THEN '7 · 30 a 60 min'        WHEN minutos < 120     THEN '8 · 1 a 2 h'        WHEN minutos < 1440    THEN '9 · 2 a 24 h'        ELSE                        'A · mais de 24 h'    END AS faixa,    COUNT(*) AS corridas,    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pctFROM duracoesGROUP BY faixaORDER BY faixa;

### Achado 4 — contagem de passageiros

In [ ]:
%sqlWITH passageiros AS (    SELECT passenger_count FROM ifood_case.trusted.ny_taxi_trip_yellow    UNION ALL    SELECT passenger_count FROM ifood_case.trusted.ny_taxi_trip_green)SELECT    CASE        WHEN passenger_count IS NULL THEN '0 · nulo'        WHEN passenger_count < 0     THEN '1 · negativo'        WHEN passenger_count = 0     THEN '2 · zero'        WHEN passenger_count <= 4    THEN '3 · 1 a 4'        WHEN passenger_count <= 6    THEN '4 · 5 a 6'        ELSE                              '5 · acima de 6'    END AS faixa,    COUNT(*) AS corridas,    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pctFROM passageirosGROUP BY faixaORDER BY faixa;

### Achado 5 — distribuição do `total_amount`Interessa saber se os valores negativos são poucos (estorno pontual) ourelevantes, e quão longa é a cauda superior — isso decide se a **média** é umamétrica honesta ou se a **mediana** conta melhor a história.

In [ ]:
%sqlSELECT    CASE        WHEN total_amount IS NULL THEN '0 · nulo'        WHEN total_amount < 0     THEN '1 · negativo'        WHEN total_amount = 0     THEN '2 · zero'        WHEN total_amount < 10    THEN '3 · até 10'        WHEN total_amount < 25    THEN '4 · 10 a 25'        WHEN total_amount < 50    THEN '5 · 25 a 50'        WHEN total_amount < 100   THEN '6 · 50 a 100'        WHEN total_amount < 1000  THEN '7 · 100 a 1.000'        ELSE                           '8 · acima de 1.000'    END AS faixa,    COUNT(*) AS corridas,    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pct,    ROUND(SUM(total_amount), 2) AS somaFROM ifood_case.trusted.ny_taxi_trip_yellowGROUP BY faixaORDER BY faixa;

In [ ]:
%sql-- Quantis: mostra o quanto a cauda puxa a média para cima.SELECT    ROUND(percentile_approx(total_amount, 0.01), 2) AS p01,    ROUND(percentile_approx(total_amount, 0.25), 2) AS p25,    ROUND(percentile_approx(total_amount, 0.50), 2) AS mediana,    ROUND(percentile_approx(total_amount, 0.75), 2) AS p75,    ROUND(percentile_approx(total_amount, 0.95), 2) AS p95,    ROUND(percentile_approx(total_amount, 0.99), 2) AS p99,    ROUND(AVG(total_amount), 2)                     AS mediaFROM ifood_case.trusted.ny_taxi_trip_yellowWHERE total_amount IS NOT NULL;

## 3. Efeito das regras de qualidadeQuanto do dado sobrevive à limpeza, e por quais motivos o resto foi barrado.

In [ ]:
%sqlSELECT rule_name, blocking, rows_evaluated, rows_failed, failure_pct, descriptionFROM ifood_case.quality.dq_resultsORDER BY blocking DESC, rows_failed DESC;

In [ ]:
%sqlSELECT    (SELECT COUNT(*) FROM ifood_case.refined.fct_taxi_trip) AS aprovadas,    (SELECT COUNT(*) FROM ifood_case.refined.rej_taxi_trip) AS em_quarentena,    ROUND(        (SELECT COUNT(*) FROM ifood_case.refined.rej_taxi_trip) * 100.0 /        ((SELECT COUNT(*) FROM ifood_case.refined.fct_taxi_trip) +         (SELECT COUNT(*) FROM ifood_case.refined.rej_taxi_trip)), 4    ) AS pct_quarentena;

## 4. Conclusões que viraram regra no pipeline| Achado | Decisão | Onde está no código ||---|---|---|| Datas fora de Jan–Mai/2023 | Quarentena (`fora_da_janela_de_analise`) | `src/quality/expectations.py` || Término anterior ao início | Quarentena (`duracao_nao_positiva`) | idem || Corridas com mais de 24 h | Quarentena (`duracao_acima_de_24h`) | idem || `total_amount` negativo | Quarentena (`total_amount_negativo`), com análise de sensibilidade nas respostas | idem || `passenger_count` nulo ou zero | **Não bloqueia** — a corrida ainda vale para receita; só as análises de passageiro filtram | idem || Média puxada pela cauda | Mediana reportada ao lado da média | `analysis/02_respostas.ipynb` |Nenhuma dessas linhas é apagada: todas ficam em `ifood_case.refined.rej_taxi_trip`com o motivo, disponíveis para reprocessamento se a regra mudar.